In [6]:
import cv2
import mediapipe as mp
from mediapipe import solutions
# removed unused import of landmark_pb2 (not needed in this cell)

# Correct MediaPipe solution access
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Open webcam stream
cap = cv2.VideoCapture(0)

# Configure Face Mesh
with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Webcam successfully opened! Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()

        if not success:
            print("Ignoring empty camera frame.")
            continue

        # Convert image to RGB
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Process image
        results = face_mesh.process(image)

        # Convert back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw face mesh
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:

                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=
                    mp_drawing_styles.get_default_face_mesh_tesselation_style()
                )

        # Show window
        cv2.imshow('MediaPipe Face Mesh Test', image)

        # Exit on q
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# Release resources
cap.release()
cv2.destroyAllWindows()

Webcam successfully opened! Press 'q' to exit.


In [9]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

# Correct MediaPipe solution access
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Define landmark indices for Left and Right Eyes
LEFT_EYE_VERT_1 = [160, 144]  # Upper and lower eyelid points 1
LEFT_EYE_VERT_2 = [158, 153]  # Upper and lower eyelid points 2
LEFT_EYE_HORIZ  = [33, 133]   # Inner and outer corners

RIGHT_EYE_VERT_1 = [385, 380] # Upper and lower eyelid points 1
RIGHT_EYE_VERT_2 = [387, 373] # Upper and lower eyelid points 2
RIGHT_EYE_HORIZ  = [362, 263] # Inner and outer corners

# EAR Threshold to determine if eye is closed
EAR_THRESHOLD = 0.23  # If EAR goes below this, eye is considered closed

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    """
    Calculates the Eye Aspect Ratio (EAR) based on Euclidean distances of landmarks.
    """
    try:
        # Extract pixel coordinates for vertical landmarks
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        
        # Extract pixel coordinates for horizontal landmarks
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        # Calculate Euclidean distances
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        
        # Apply the EAR formula
        ear = (dist_v1 + dist_v2) / (2.0 * dist_h)
        return ear
    except Exception as e:
        return 0.0

# Open webcam stream
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("EAR Tracking successfully started! Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        
        # Convert image to RGB for MediaPipe processing
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        # Convert back to BGR for drawing
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # Calculate EAR for both eyes
                left_ear = calculate_ear(face_landmarks.landmark, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(face_landmarks.landmark, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                
                # Average EAR value
                avg_ear = (left_ear + right_ear) / 2.0
                
                # Display EAR value on the video screen
                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                
                # Visual Alert Logic if EAR drops below threshold
                if avg_ear < EAR_THRESHOLD:
                    cv2.putText(image, "DROWSY ALERT!", (30, 100), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                
                # Draw only the eye contours to keep it clean and professional
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        # Show visualization window
        cv2.imshow('Driver Safety AI - EAR Tracking MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

EAR Tracking successfully started! Press 'q' to exit.


In [10]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

# Initialize MediaPipe modules
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Landmark indices for Left and Right Eyes
LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]

RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

# ---- TUNED DROWSINESS CONFIGURATION ----
EAR_THRESHOLD = 0.21          # Tuned based on your squinting threshold
CONSECUTIVE_FRAMES_LIMIT = 15 # Eye must be closed for at least 15 consecutive frames (~0.5 - 1 second) to alert
FRAME_COUNTER = 0             # Keeps track of consecutive closed-eye frames

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    """Calculates the Eye Aspect Ratio (EAR) based on Euclidean distances."""
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

# Open webcam stream
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Advanced Time-Series EAR Tracking Started... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                left_ear = calculate_ear(face_landmarks.landmark, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(face_landmarks.landmark, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                
                avg_ear = (left_ear + right_ear) / 2.0
                
                # Check if EAR is below threshold
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1 # Increment frame count if eyes are closed
                else:
                    FRAME_COUNTER = 0 # RESET counter instantly if eyes are opened even for a single frame

                # Display real-time EAR and active frame buffer metrics
                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                cv2.putText(image, f"Closed Frames: {FRAME_COUNTER}", (30, 90), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
                
                # Trigger alert ONLY if the eyes are closed continuously beyond the frame limit
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "!!! DROWSY ALERT !!!", (30, 140), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
                
                # Draw facial contours
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Advanced EAR MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Advanced Time-Series EAR Tracking Started... Press 'q' to exit.


In [12]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

# Initialize MediaPipe Face Mesh
mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Eye Landmark Indices
LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]
RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

# Configuration Thresholds
EAR_THRESHOLD = 0.21
CONSECUTIVE_FRAMES_LIMIT = 15
FRAME_COUNTER = 0

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    """Calculates Eye Aspect Ratio (EAR) using Euclidean distances."""
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

# Open webcam stream
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Drowsiness + Distraction Tracker Started... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                landmarks = face_landmarks.landmark
                
                # ---------------- 1. DROWSINESS DETECTION (EAR) ----------------
                left_ear = calculate_ear(landmarks, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1
                else:
                    FRAME_COUNTER = 0

                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "🚨 DROWSY ALERT! 🚨", (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

                # ---------------- 2. DISTRACTION DETECTION (HEAD POSE) ----------------
                # Extract 2D coordinates of specific facial anchor points
                image_points = np.array([
                    [landmarks[1].x * img_w, landmarks[1].y * img_h],     # Nose tip
                    [landmarks[152].x * img_w, landmarks[152].y * img_h], # Chin
                    [landmarks[33].x * img_w, landmarks[33].y * img_h],   # Left eye outer corner
                    [landmarks[263].x * img_w, landmarks[263].y * img_h], # Right eye outer corner
                    [landmarks[61].x * img_w, landmarks[61].y * img_h],   # Left mouth corner
                    [landmarks[291].x * img_w, landmarks[291].y * img_h]  # Right mouth corner
                ], dtype="double")

                # Standard 3D generic head model coordinates
                model_points = np.array([
                    (0.0, 0.0, 0.0),             # Nose tip
                    (0.0, -330.0, -65.0),        # Chin
                    (-225.0, 170.0, -135.0),     # Left eye outer corner
                    (225.0, 170.0, -135.0),      # Right eye outer corner
                    (-150.0, -150.0, -125.0),    # Left mouth corner
                    (150.0, -150.0, -125.0)      # Right mouth corner
                ])

                # Camera internals (Approximated focal length and center)
                focal_length = img_w
                center = (img_w / 2, img_h / 2)
                camera_matrix = np.array([
                    [focal_length, 0, center[0]],
                    [0, focal_length, center[1]],
                    [0, 0, 1]
                ], dtype="double")
                
                dist_coeffs = np.zeros((4, 1)) # Assuming no lens distortion

                # Solve Perspective-n-Point to get rotation vector
                success_pnp, rotation_vector, translation_vector = cv2.solvePnP(
                    model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE
                )

                # Convert rotation vector to rotation matrix, then to Euler angles
                rmat, _ = cv2.Rodrigues(rotation_vector)
                angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)
                
                # Extract pitch (up/down) and yaw (left/right) angles
                pitch = angles[0] * 360
                yaw = angles[1] * 360

                # Check if head is turned too far away from the road center
                if abs(yaw) > 18 or pitch < -10:
                    cv2.putText(image, "⚠️ DISTRACTED! ⚠️", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)
                
                # Draw facial contours to render the tracking web mesh
                mp_drawing.draw_landmarks(
                    image=image, landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS, landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Full MVP (EAR + Head Pose)', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Drowsiness + Distraction Tracker Started... Press 'q' to exit.


In [3]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Eye indices for EAR
LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]
RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

EAR_THRESHOLD = 0.21
CONSECUTIVE_FRAMES_LIMIT = 15
FRAME_COUNTER = 0

# ---- DYNAMIC CALIBRATION VARIABLES ----
CALIBRATION_FRAMES = 30  # Number of frames used to calculate the baseline pose
calibrated = False       # Flag to check if calibration is done
frame_count = 0          # Internal counter for calibration frames
baseline_pitch = 0.0     # Reference pitch angle when looking straight
baseline_yaw = 0.0       # Reference yaw angle when looking straight

pitch_accumulator = []
yaw_accumulator = []

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Dynamic Calibration System Active... Look straight at the camera initially.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                landmarks = face_landmarks.landmark
                
                # ---------------- 1. DROWSINESS (EAR) ----------------
                left_ear = calculate_ear(landmarks, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1
                else:
                    FRAME_COUNTER = 0

                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "🚨 DROWSY ALERT! 🚨", (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

                # ---------------- 2. HEAD POSE COMPUTATION ----------------
                image_points = np.array([
                    [landmarks[1].x * img_w, landmarks[1].y * img_h],     # Nose tip
                    [landmarks[152].x * img_w, landmarks[152].y * img_h], # Chin
                    [landmarks[33].x * img_w, landmarks[33].y * img_h],   # Left eye outer corner
                    [landmarks[263].x * img_w, landmarks[263].y * img_h], # Right eye outer corner
                    [landmarks[61].x * img_w, landmarks[61].y * img_h],   # Left mouth corner
                    [landmarks[291].x * img_w, landmarks[291].y * img_h]  # Right mouth corner
                ], dtype="double")

                model_points = np.array([
                    (0.0, 0.0, 0.0),             # Nose tip
                    (0.0, -330.0, -65.0),        # Chin
                    (-225.0, 170.0, -135.0),     # Left eye outer corner
                    (225.0, 170.0, -135.0),      # Right eye outer corner
                    (-150.0, -150.0, -125.0),    # Left mouth corner
                    (150.0, -150.0, -125.0)      # Right mouth corner
                ])

                focal_length = img_w
                center = (img_w / 2, img_h / 2)
                camera_matrix = np.array([
                    [focal_length, 0, center[0]],
                    [0, focal_length, center[1]],
                    [0, 0, 1]
                ], dtype="double")
                
                dist_coeffs = np.zeros((4, 1))

                success_pnp, rotation_vector, translation_vector = cv2.solvePnP(
                    model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE
                )

                rmat, _ = cv2.Rodrigues(rotation_vector)
                angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)
                
                raw_pitch = angles[0] * 360
                raw_yaw = angles[1] * 360

                # ---------------- 3. CALIBRATION LOGIC ----------------
                if not calibrated:
                    frame_count += 1
                    pitch_accumulator.append(raw_pitch)
                    yaw_accumulator.append(raw_yaw)
                    
                    cv2.putText(image, f"CALIBRATING BASELINE POSE... ({frame_count}/{CALIBRATION_FRAMES})", 
                                (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                    
                    if frame_count >= CALIBRATION_FRAMES:
                        baseline_pitch = np.mean(pitch_accumulator)
                        baseline_yaw = np.mean(yaw_accumulator)
                        calibrated = True
                        print(f"Calibration Complete! Baseline Pitch: {baseline_pitch:.2f}, Baseline Yaw: {baseline_yaw:.2f}")
                else:
                    # Calculate relative deviation angles based on the saved baseline
                    relative_pitch = raw_pitch - baseline_pitch
                    relative_yaw = raw_yaw - baseline_yaw

                    # Display calibrated angles on the screen
                    cv2.putText(image, f"Relative Pitch: {relative_pitch:.1f}", (img_w - 280, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                    cv2.putText(image, f"Relative Yaw: {relative_yaw:.1f}", (img_w - 280, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

                    # ---- STRICT DISTRACTION DETECTION LIMITS ----
                    # Since it is relative, looking straight means ~0.0 degrees deviation.
                    # Looking left/right over 18 degrees or down more than 12 degrees triggers alert.
                    if abs(relative_yaw) > 18 or relative_pitch < -12:
                        cv2.putText(image, "⚠️ DISTRACTED! ⚠️", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 3)
                
                # Draw facial contours
                mp_drawing.draw_landmarks(
                    image=image, landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS, landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Calibrated Hybrid MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Dynamic Calibration System Active... Look straight at the camera initially.
Calibration Complete! Baseline Pitch: 78.33, Baseline Yaw: -3833.25


In [15]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

# Eye indices for EAR
LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]
RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

# ---- FINAL OPTIMIZED THRESHOLDS ----
EAR_THRESHOLD = 0.22          # Clean threshold for your eyes
CONSECUTIVE_FRAMES_LIMIT = 12 # About 0.5 seconds of continuous closure
FRAME_COUNTER = 0

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Stable Hybrid Tracker Started... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                landmarks = face_landmarks.landmark
                
                # ---------------- 1. DROWSINESS DETECTION (EAR) ----------------
                left_ear = calculate_ear(landmarks, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1
                else:
                    FRAME_COUNTER = 0

                # Display EAR status on screen
                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "🚨 DROWSY ALERT! 🚨", (30, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

                # ---------------- 2. LIGHTWEIGHT DISTRACTION DETECTION ----------------
                # Get 2D pixel positions of Nose tip, Left Eye Profile, Right Eye Profile
                nose = np.array([landmarks[1].x * img_w, landmarks[1].y * img_h])
                left_bound = np.array([landmarks[234].x * img_w, landmarks[234].y * img_h]) # Leftmost cheek profile
                right_bound = np.array([landmarks[454].x * img_w, landmarks[454].y * img_h]) # Rightmost cheek profile
                forehead = np.array([landmarks[10].x * img_w, landmarks[10].y * img_h])
                chin = np.array([landmarks[152].x * img_w, landmarks[152].y * img_h])

                # Calculate horizontal symmetry ratio (Left-to-Right face split)
                dist_left = np.linalg.norm(nose - left_bound)
                dist_right = np.linalg.norm(nose - right_bound)
                total_width = dist_left + dist_right
                
                # Turn ratio indicates left/right look deviation
                turn_ratio = dist_left / total_width if total_width > 0 else 0.5
                
                # Vertical ratio (Forehead to Nose vs Nose to Chin) to detect looking down
                dist_up = np.linalg.norm(forehead - nose)
                dist_down = np.linalg.norm(chin - nose)
                vertical_ratio = dist_up / dist_down if dist_down > 0 else 1.0

                # Check thresholds based on horizontal profile ratio or deep look down
                # If turn_ratio is near 0.5, you are looking center. If < 0.33 or > 0.67, you are looking sideways.
                if turn_ratio < 0.33 or turn_ratio > 0.67 or vertical_ratio < 0.65:
                    cv2.putText(image, "⚠️ DISTRACTED! ⚠️", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 3)

                # Draw clean contours
                mp_drawing.draw_landmarks(
                    image=image, landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS, landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Clean Stable MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Stable Hybrid Tracker Started... Press 'q' to exit.


In [16]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]
RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

# ---- DROWSINESS CONFIG ----
EAR_THRESHOLD = 0.22          
CONSECUTIVE_FRAMES_LIMIT = 12 
FRAME_COUNTER = 0

# ---- NEW: DISTRACTION TIME BUFFER CONFIG ----
# Only alerts if driver looks away continuously for more than 25 frames (~1 second)
DISTRACT_FRAME_LIMIT = 25     
DISTRACT_COUNTER = 0          

cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Production-Ready Driver Monitor Active... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                landmarks = face_landmarks.landmark
                
                # ---------------- 1. DROWSINESS (EAR) ----------------
                left_ear = calculate_ear(landmarks, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1
                else:
                    FRAME_COUNTER = 0

                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "🚨 DROWSY ALERT! 🚨", (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

                # ---------------- 2. TUNED DISTRACTION DETECTION ----------------
                nose = np.array([landmarks[1].x * img_w, landmarks[1].y * img_h])
                left_bound = np.array([landmarks[234].x * img_w, landmarks[234].y * img_h]) 
                right_bound = np.array([landmarks[454].x * img_w, landmarks[454].y * img_h]) 
                forehead = np.array([landmarks[10].x * img_w, landmarks[10].y * img_h])
                chin = np.array([landmarks[152].x * img_w, landmarks[152].y * img_h])

                dist_left = np.linalg.norm(nose - left_bound)
                dist_right = np.linalg.norm(nose - right_bound)
                total_width = dist_left + dist_right
                turn_ratio = dist_left / total_width if total_width > 0 else 0.5
                
                dist_up = np.linalg.norm(forehead - nose)
                dist_down = np.linalg.norm(chin - nose)
                vertical_ratio = dist_up / dist_down if dist_down > 0 else 1.0

                # Fine-tuned thresholds: Slightly expanded to allow natural mirror glances
                if turn_ratio < 0.28 or turn_ratio > 0.72 or vertical_ratio < 0.60:
                    DISTRACT_COUNTER += 1 # Eyes/Head away, increment counter
                else:
                    DISTRACT_COUNTER = 0  # Looking straight, reset instantly

                # Render active counter for debugging
                cv2.putText(image, f"Look Away Frames: {DISTRACT_COUNTER}", (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

                # TRIGGER ALERT ONLY IF LOOKING AWAY CONTINUOUSLY
                if DISTRACT_COUNTER >= DISTRACT_FRAME_LIMIT:
                    cv2.putText(image, "⚠️ DISTRACTED! ⚠️", (30, 210), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 3)

                mp_drawing.draw_landmarks(
                    image=image, landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS, landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Full Stable MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Production-Ready Driver Monitor Active... Press 'q' to exit.


In [17]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]
RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

# ---- 1. DROWSINESS CONFIG (UNTOUCHED - EXACTLY AS BEFORE) ----
EAR_THRESHOLD = 0.22          
CONSECUTIVE_FRAMES_LIMIT = 12 
FRAME_COUNTER = 0

# ---- 2. DISTRACTION CONFIG (TUNED AS REQUESTED) ----
# Increased from 25 to 45 frames (~1.5 to 2 seconds) for a smoother drive experience
DISTRACT_FRAME_LIMIT = 45     
DISTRACT_COUNTER = 0          

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("Optimized Driver Monitor Active... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                landmarks = face_landmarks.landmark
                
                # ---------------- 1. DROWSINESS (EAR) ----------------
                left_ear = calculate_ear(landmarks, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1
                else:
                    FRAME_COUNTER = 0

                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "🚨 DROWSY ALERT! 🚨", (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

                # ---------------- 2. HIGH-TOLERANCE DISTRACTION ----------------
                nose = np.array([landmarks[1].x * img_w, landmarks[1].y * img_h])
                left_bound = np.array([landmarks[234].x * img_w, landmarks[234].y * img_h]) 
                right_bound = np.array([landmarks[454].x * img_w, landmarks[454].y * img_h]) 
                forehead = np.array([landmarks[10].x * img_w, landmarks[10].y * img_h])
                chin = np.array([landmarks[152].x * img_w, landmarks[152].y * img_h])

                dist_left = np.linalg.norm(nose - left_bound)
                dist_right = np.linalg.norm(nose - right_bound)
                total_width = dist_left + dist_right
                turn_ratio = dist_left / total_width if total_width > 0 else 0.5
                
                dist_up = np.linalg.norm(forehead - nose)
                dist_down = np.linalg.norm(chin - nose)
                vertical_ratio = dist_up / dist_down if dist_down > 0 else 1.0

                # Slightly loosened thresholds (0.26 and 0.74) to ignore subtle head movements
                if turn_ratio < 0.26 or turn_ratio > 0.74 or vertical_ratio < 0.58:
                    DISTRACT_COUNTER += 1 
                else:
                    DISTRACT_COUNTER = 0  

                # Display the current look away frame counter on screen
                cv2.putText(image, f"Look Away Frames: {DISTRACT_COUNTER}/{DISTRACT_FRAME_LIMIT}", (30, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

                # Trigger alert only if the driver crosses the 45-frame limit
                if DISTRACT_COUNTER >= DISTRACT_FRAME_LIMIT:
                    cv2.putText(image, "⚠️ DISTRACTED! ⚠️", (30, 210), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 3)

                mp_drawing.draw_landmarks(
                    image=image, landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS, landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Full Stable MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Optimized Driver Monitor Active... Press 'q' to exit.


In [4]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe import solutions

mp_face_mesh = solutions.face_mesh
mp_drawing = solutions.drawing_utils
mp_drawing_styles = solutions.drawing_styles

LEFT_EYE_VERT_1 = [160, 144]
LEFT_EYE_VERT_2 = [158, 153]
LEFT_EYE_HORIZ  = [33, 133]
RIGHT_EYE_VERT_1 = [385, 380]
RIGHT_EYE_VERT_2 = [387, 373]
RIGHT_EYE_HORIZ  = [362, 263]

EAR_THRESHOLD = 0.21         
CONSECUTIVE_FRAMES_LIMIT = 15
FRAME_COUNTER = 0

DISTRACT_FRAME_LIMIT = 45     
DISTRACT_COUNTER = 0          

def calculate_ear(landmarks, vert_1, vert_2, horiz, img_w, img_h):
    try:
        p_v1_top = np.array([landmarks[vert_1[0]].x * img_w, landmarks[vert_1[0]].y * img_h])
        p_v1_bot = np.array([landmarks[vert_1[1]].x * img_w, landmarks[vert_1[1]].y * img_h])
        p_v2_top = np.array([landmarks[vert_2[0]].x * img_w, landmarks[vert_2[0]].y * img_h])
        p_v2_bot = np.array([landmarks[vert_2[1]].x * img_w, landmarks[vert_2[1]].y * img_h])
        p_h_left  = np.array([landmarks[horiz[0]].x * img_w, landmarks[horiz[0]].y * img_h])
        p_h_right = np.array([landmarks[horiz[1]].x * img_w, landmarks[horiz[1]].y * img_h])
        
        dist_v1 = np.linalg.norm(p_v1_top - p_v1_bot)
        dist_v2 = np.linalg.norm(p_v2_top - p_v2_bot)
        dist_h  = np.linalg.norm(p_h_left - p_h_right)
        return (dist_v1 + dist_v2) / (2.0 * dist_h)
    except Exception:
        return 0.0

cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    print("High-Accuracy Vertical Monitor Active... Press 'q' to exit.")

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        img_h, img_w, _ = image.shape
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                landmarks = face_landmarks.landmark
                
                # ---------------- 1. DROWSINESS (EAR) ----------------
                left_ear = calculate_ear(landmarks, LEFT_EYE_VERT_1, LEFT_EYE_VERT_2, LEFT_EYE_HORIZ, img_w, img_h)
                right_ear = calculate_ear(landmarks, RIGHT_EYE_VERT_1, RIGHT_EYE_VERT_2, RIGHT_EYE_HORIZ, img_w, img_h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < EAR_THRESHOLD:
                    FRAME_COUNTER += 1
                else:
                    FRAME_COUNTER = 0

                cv2.putText(image, f"EAR: {avg_ear:.2f}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                if FRAME_COUNTER >= CONSECUTIVE_FRAMES_LIMIT:
                    cv2.putText(image, "DROWSY ALERT!", (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

                # ---------------- 2. HIGH-ACCURACY DISTRACTION ----------------
                nose = np.array([landmarks[1].x * img_w, landmarks[1].y * img_h])
                left_bound = np.array([landmarks[234].x * img_w, landmarks[234].y * img_h]) 
                right_bound = np.array([landmarks[454].x * img_w, landmarks[454].y * img_h]) 
                
                # NEW LANDMARKS FOR BETTER VERTICAL TRACKING
                forehead_mid = np.array([landmarks[9].x * img_w, landmarks[9].y * img_h]) # Lower forehead
                chin_top = np.array([landmarks[11].x * img_w, landmarks[11].y * img_h])     # Upper chin/lip area

                # Horizontal turn math
                dist_left = np.linalg.norm(nose - left_bound)
                dist_right = np.linalg.norm(nose - right_bound)
                total_width = dist_left + dist_right
                turn_ratio = dist_left / total_width if total_width > 0 else 0.5
                
                # NEW: Highly sensitive vertical ratio calculation
                dist_up = np.linalg.norm(forehead_mid - nose)
                dist_down = np.linalg.norm(chin_top - nose)
                vertical_ratio = dist_up / dist_down if dist_down > 0 else 1.0

                # Render vertical ratio live on screen for quick verification
                cv2.putText(image, f"V-Ratio: {vertical_ratio:.2f}", (30, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # Tuned condition: If V-Ratio drops below 0.88, it means head is tilting down (looking at phone)
                if turn_ratio < 0.26 or turn_ratio > 0.74 or vertical_ratio < 0.88:
                    DISTRACT_COUNTER += 1 
                else:
                    DISTRACT_COUNTER = 0  

                cv2.putText(image, f"Look Away Frames: {DISTRACT_COUNTER}/{DISTRACT_FRAME_LIMIT}", (30, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

                if DISTRACT_COUNTER >= DISTRACT_FRAME_LIMIT:
                    cv2.putText(image, "DISTRACTED!", (30, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 3)

                mp_drawing.draw_landmarks(
                    image=image, landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS, landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )

        cv2.imshow('Driver Safety AI - Full Stable MVP', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

High-Accuracy Vertical Monitor Active... Press 'q' to exit.
